# Week 2c: evaluation with pycocotools, per class and per object size

The Ultralytics validator gives one AP per class. The research plan asks for more: AP per class **and** per object size (small < 32x32 px, medium < 96x96 px, large), and a 95 % interval for every number, because forklift has only 124 boxes in val.

This notebook runs every trained model in `/kaggle/input` on the val images (subsets 1 and 4), scores the detections with the official COCO evaluation (pycocotools) and computes bootstrap intervals by resampling whole images 1,000 times.

Before running: Accelerator = GPU T4, Internet = on. With **+ Add Input** add the LOCO image dataset and the outputs of the earlier notebooks (they contain `yolo11n_960_best.pt`, `rtdetr_l_640_best.pt` and, after notebook 03, the 640 px run). Then Save Version > **Save & Run All (Commit)**.

Output: `out/coco_ap.csv`, `out/coco_ap_table.md`, `out/ap_by_size.png`, one detection file per model, zipped as `results_04.zip`.

In [ ]:
!pip install -q ultralytics pycocotools

In [ ]:
# Annotations (COCO format) from the LOCO repository
!git clone -q --depth 1 https://github.com/tum-fml/loco /kaggle/working/loco
!ls /kaggle/working/loco/rgb

In [ ]:
import zipfile, tarfile
from pathlib import Path

# Unpack the image dataset if it was uploaded as archives, then index every jpg by file name.
extract_dir = Path("/kaggle/working/loco_images")
for a in [p for p in Path("/kaggle/input").rglob("*") if p.suffix.lower() in {".zip", ".tar", ".gz", ".tgz"}]:
    target = extract_dir / a.stem
    target.mkdir(parents=True, exist_ok=True)
    if a.suffix.lower() == ".zip":
        zipfile.ZipFile(a).extractall(target)
    else:
        tarfile.open(a).extractall(target)
JPG = {}
for root in [Path("/kaggle/input"), extract_dir]:
    if root.exists():
        for p in root.rglob("*.jpg"):
            JPG.setdefault(p.name, p)
print(len(JPG), "jpg files indexed")

## Ground truth for val

Same split and the same five classes as the training notebooks. Boxes are clipped to the image, like in `coco_to_yolo.py`, and the area of a box is its width times its height, so "small" means smaller than 32x32 pixels exactly as in the README statistics.

In [ ]:
import json, contextlib, io
import numpy as np
from pycocotools.coco import COCO

CLASSES = {3: "small_load_carrier", 5: "forklift", 7: "pallet", 10: "stillage", 11: "pallet_truck"}
CAT_IDS = list(CLASSES)          # YOLO class index i -> LOCO category id CAT_IDS[i]
VAL_FILES = ["loco-sub1-v1-val.json", "loco-sub4-v1-val.json"]

images, annotations, path_of = [], [], {}
for f in VAL_FILES:
    d = json.load(open(f"/kaggle/working/loco/rgb/{f}"))
    old_to_new = {}
    for im in d["images"]:
        name = Path(im["file_name"]).name
        assert name in JPG, f"image not found: {name}"
        new_id = len(images) + 1
        old_to_new[im["id"]] = new_id
        images.append(dict(id=new_id, file_name=name, width=im["width"], height=im["height"]))
        path_of[new_id] = JPG[name]
    size = {old_to_new[im["id"]]: (im["width"], im["height"]) for im in d["images"]}
    for a in d["annotations"]:
        if a["category_id"] not in CLASSES or a["image_id"] not in old_to_new:
            continue
        iid = old_to_new[a["image_id"]]
        W, H = size[iid]
        x, y, w, h = a["bbox"]
        x0, y0, x1, y1 = max(x, 0), max(y, 0), min(x + w, W), min(y + h, H)
        if x1 <= x0 or y1 <= y0:
            continue
        annotations.append(dict(id=len(annotations) + 1, image_id=iid, category_id=a["category_id"],
                                bbox=[x0, y0, x1 - x0, y1 - y0], area=(x1 - x0) * (y1 - y0), iscrowd=0))

gt = COCO()
gt.dataset = dict(images=images, annotations=annotations,
                  categories=[dict(id=c, name=n) for c, n in CLASSES.items()])
with contextlib.redirect_stdout(io.StringIO()):
    gt.createIndex()

areas = np.array([a["area"] for a in annotations])
cats = np.array([a["category_id"] for a in annotations])
print(len(images), "val images,", len(annotations), "boxes")
for c, n in CLASSES.items():
    s = areas[cats == c]
    print(f"{n:20s} {len(s):6d} boxes   small {np.mean(s < 32**2):.1%}   medium {np.mean((s >= 32**2) & (s < 96**2)):.1%}   large {np.mean(s >= 96**2):.1%}")

## Helper functions

`ap()` repeats the computation of `COCOeval.accumulate` for one class and one size range, with one extra: every image can get a weight. With all weights equal to 1 it gives exactly the COCO result (checked against pycocotools on synthetic data: the difference was below 1e-15). A bootstrap sample is then just a weight vector, so the 1,000 resamples do not need 1,000 COCO evaluations.

In [ ]:
import numpy as np


def collect(E, k, a, max_det=None):
    """Detections of one class (index k) and one area range (index a) from a COCOeval run,
    sorted by score over all images, as COCOeval.accumulate does it."""
    p = E.params
    n_img, n_area = len(p.imgIds), len(p.areaRng)
    max_det = max_det or p.maxDets[-1]
    rows = E.evalImgs[k * n_area * n_img + a * n_img: k * n_area * n_img + (a + 1) * n_img]
    scores, img, match, ignore = [], [], [], []
    npig = np.zeros(n_img)
    for i, e in enumerate(rows):
        if e is None:
            continue
        s = np.asarray(e['dtScores'][:max_det])
        scores.append(s)
        img.append(np.full(len(s), i))
        match.append(e['dtMatches'][:, :max_det])
        ignore.append(e['dtIgnore'][:, :max_det])
        npig[i] = np.count_nonzero(np.asarray(e['gtIgnore']) == 0)
    n_thr = len(p.iouThrs)
    if scores:
        scores = np.concatenate(scores)
        order = np.argsort(-scores, kind='mergesort')
        img = np.concatenate(img)[order]
        match = np.concatenate(match, axis=1)[:, order].astype(bool)
        ignore = np.concatenate(ignore, axis=1)[:, order].astype(bool)
    else:
        img = np.zeros(0, int)
        match = ignore = np.zeros((n_thr, 0), bool)
    return dict(img=img, tp=match & ~ignore, fp=~match & ~ignore, npig=npig,
                rec_thrs=np.asarray(p.recThrs))


def ap(part, weights=None, thr=None):
    """COCO AP (mean over IoU thresholds and 101 recall points) for one class and area range.
    weights[i] = how often image i is in the sample (all ones = the normal COCO result).
    thr = index of one IoU threshold (0 = IoU 0.5) instead of the mean over all ten."""
    n_img = len(part['npig'])
    w_img = np.ones(n_img) if weights is None else np.asarray(weights, float)
    npig = float((w_img * part['npig']).sum())
    if npig == 0:
        return np.nan
    if part['img'].size == 0:
        return 0.0
    w = w_img[part['img']]
    rows = slice(None) if thr is None else slice(thr, thr + 1)
    tp = np.cumsum(part['tp'][rows] * w, axis=1)
    fp = np.cumsum(part['fp'][rows] * w, axis=1)
    rc = tp / npig
    pr = tp / (tp + fp + np.spacing(1))
    pr = np.maximum.accumulate(pr[:, ::-1], axis=1)[:, ::-1]
    out = []
    for t in range(rc.shape[0]):
        idx = np.searchsorted(rc[t], part['rec_thrs'], side='left')
        ok = idx < rc.shape[1]
        q = np.zeros(len(idx))
        q[ok] = pr[t, idx[ok]]
        out.append(q.mean())
    return float(np.mean(out))


def bootstrap(parts, n_img, n_boot=1000, seed=0):
    """AP of every part on n_boot resamples of the images (drawn with replacement)."""
    rng = np.random.default_rng(seed)
    res = {key: np.empty(n_boot) for key in parts}
    for b in range(n_boot):
        wts = np.bincount(rng.integers(0, n_img, n_img), minlength=n_img)
        for key, part in parts.items():
            res[key][b] = ap(part, wts)
    return res


def interval(samples):
    return float(np.nanpercentile(samples, 2.5)), float(np.nanpercentile(samples, 97.5))

## Detections

Every `.pt` file in the inputs is run once. The input size is read from the file name (`..._640_...` or `..._960_...`), so each model is evaluated at the size it was trained with. `conf=0.001` and `max_det=500` are the same settings as in the Ultralytics validation.

In [ ]:
import re, csv, gzip
import torch
from ultralytics import YOLO, RTDETR

OUT = Path("/kaggle/working/out")
OUT.mkdir(exist_ok=True)
weights = sorted(p for p in Path("/kaggle/input").rglob("*.pt"))
print("\n".join(str(w) for w in weights))
assert weights, "no .pt files found: add the outputs of the training notebooks with + Add Input"

def detect(weight):
    imgsz = int(re.search(r"_(640|960)", weight.name).group(1))
    model = (RTDETR if "rtdetr" in weight.name else YOLO)(str(weight))
    ids = sorted(path_of)
    dets = []
    for start in range(0, len(ids), 32):
        chunk = ids[start:start + 32]
        results = model.predict([str(path_of[i]) for i in chunk], imgsz=imgsz, conf=0.001, max_det=500,
                                verbose=False, device=0 if torch.cuda.is_available() else "cpu")
        for iid, r in zip(chunk, results):
            b = r.boxes
            for (x0, y0, x1, y1), s, c in zip(b.xyxy.tolist(), b.conf.tolist(), b.cls.tolist()):
                dets.append(dict(image_id=iid, category_id=CAT_IDS[int(c)],
                                 bbox=[x0, y0, x1 - x0, y1 - y0], score=s))
    with gzip.open(OUT / f"detections_{weight.stem}.csv.gz", "wt", newline="") as f:
        w = csv.writer(f)
        w.writerow(["image_id", "category_id", "x", "y", "w", "h", "score"])
        for d in dets:
            w.writerow([d["image_id"], d["category_id"], *[round(v, 2) for v in d["bbox"]], round(d["score"], 5)])
    return dets

DETS = {}
for w in weights:
    DETS[w.stem] = detect(w)
    print(w.stem, len(DETS[w.stem]), "detections")

## COCO evaluation and bootstrap intervals

In [ ]:
import pandas as pd
from pycocotools.cocoeval import COCOeval

AREAS = ["all", "small", "medium", "large"]
N_BOOT = 1000
rows = []
for name, dets in DETS.items():
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt.loadRes(dets)
        E = COCOeval(gt, dt, "bbox")
        E.params.maxDets = [1, 100, 500]
        E.evaluate(); E.accumulate(); E.summarize()
    parts = {(k, a): collect(E, k, a) for k in range(len(CAT_IDS)) for a in range(len(AREAS))}
    boots = bootstrap(parts, n_img=len(E.params.imgIds), n_boot=N_BOOT)
    for a, area in enumerate(AREAS):
        per_class = []
        for k, cid in enumerate(E.params.catIds):
            part = parts[(k, a)]
            lo, hi = interval(boots[(k, a)])
            rows.append(dict(model=name, cls=CLASSES[cid], area=area, boxes=int(part["npig"].sum()),
                             AP=ap(part), AP50=ap(part, thr=0), AP_lo=lo, AP_hi=hi))
            per_class.append(boots[(k, a)])
        m = np.nanmean(np.vstack(per_class), axis=0)
        lo, hi = interval(m)
        rows.append(dict(model=name, cls="mean over classes", area=area,
                         boxes=int(sum(parts[(k, a)]["npig"].sum() for k in range(len(CAT_IDS)))),
                         AP=float(np.nanmean([ap(parts[(k, a)]) for k in range(len(CAT_IDS))])),
                         AP50=float(np.nanmean([ap(parts[(k, a)], thr=0) for k in range(len(CAT_IDS))])),
                         AP_lo=lo, AP_hi=hi))
    print(f"{name}: COCO mAP {E.stats[0]:.3f} (small {E.stats[3]:.3f}, medium {E.stats[4]:.3f}, large {E.stats[5]:.3f})")

res = pd.DataFrame(rows).round(3)
res.to_csv(OUT / "coco_ap.csv", index=False)

def md_table(p):
    head = "| class | " + " | ".join(p.columns) + " |"
    sep = "|---|" + "---:|" * len(p.columns)
    body = ["| " + idx + " | " + " | ".join(str(v) for v in row) + " |" for idx, row in zip(p.index, p.values)]
    return "\n".join([head, sep] + body)

def cell(r):
    return f"{r.AP:.3f} [{r.AP_lo:.3f}, {r.AP_hi:.3f}]" if r.boxes else "-"
table = []
for name in DETS:
    t = res[res.model == name].copy()
    t["value"] = [cell(r) for r in t.itertuples()]
    p = t.pivot(index="cls", columns="area", values="value")[AREAS]
    table.append(f"### {name}\n\nAP50-95 with 95 % bootstrap interval ({N_BOOT} resamples of the val images)\n\n" + md_table(p))
(OUT / "coco_ap_table.md").write_text("\n\n".join(table) + "\n")
print("\n\n".join(table))

In [ ]:
import matplotlib.pyplot as plt

names = list(DETS)
fig, axes = plt.subplots(1, len(names), figsize=(6 * len(names), 4), sharey=True, squeeze=False)
for ax, name in zip(axes[0], names):
    t = res[(res.model == name) & (res.cls != "mean over classes") & (res.area != "all")]
    classes = list(CLASSES.values())
    x = np.arange(len(classes))
    for j, area in enumerate(["small", "medium", "large"]):
        s = t[t.area == area].set_index("cls").reindex(classes)
        err = np.vstack([s.AP - s.AP_lo, s.AP_hi - s.AP]).clip(min=0)
        ax.bar(x + (j - 1) * 0.27, s.AP, 0.27, yerr=err, capsize=2, label=area)
    ax.set_xticks(x, [c.replace("_", "\n") for c in classes])
    ax.set_title(name); ax.set_ylabel("AP50-95")
axes[0][0].legend()
plt.tight_layout(); plt.savefig(OUT / "ap_by_size.png", dpi=120); plt.show()

## Findings

Write three findings in your own words, with the numbers from the table (this closes week 2 in the README):

1.
2.
3.

In [ ]:
import shutil, os
shutil.rmtree("/kaggle/working/loco", ignore_errors=True)
shutil.rmtree("/kaggle/working/loco_images", ignore_errors=True)
shutil.make_archive("/kaggle/working/results_04", "zip", OUT)
print(sorted(os.listdir(OUT)))